Guide: Point Sources
--------------------

The examples covered so far have focused on strongly lensed galaxies, where extended surface brightness is warped into
stunning giant arcs and Einstein rings visible in high-quality telescope images. For these extended sources, light
profile objects—such as analytic Sersic profiles—are used to represent their surface brightness.

However, some observed sources are extremely small, spanning just light weeks or days across. In these cases, only the
source’s central point of light is detected in each multiple image. Such sources are called **point sources**,
which typically include quasars, supernovae, or stars.

Strictly speaking, a point source does have a finite size—on the order of light weeks or days—but it is considered a
point source because its size is orders of magnitude smaller than the resolution of the telescope. As a result, it
appears as a single point of light, with all the flux of each multiple image effectively contained within a single pixel.

Point sources affect lensing calculations differently than extended sources, requiring dedicated methods and
functionality. This functionality is described here and used throughout the `point_source` simulation and modeling
examples.

If you are new to analyzing strong lenses with point sources, this guide is the ideal place to start!

__Contents__

- **Lensed Point Source:** To begin, we create a strong lens image using an isothermal mass model and a source with a compact.
- **Point Source:** The image above visually illustrates where the source’s light is traced on the image plane.
- **Point Solver:** For a point source, our goal is to find the (y, x) coordinates in the image plane that map directly.
- **Number of Solutions:** The number of solutions (e.g.
- **Solving the Lens Equation:** In the literature, the process of finding the multiple images of a source in the image-plane is.
- **Triangle Tracing:** Computing the multiple image positions of a point source is a non-linear problem.
- **Dataset:** Load and plot the strong lens dataset.
- **Name Pairing:** The names of the point-source datasets have an even more important role, the names are used to pair.
- **Fitting:** Fit the lens model to the dataset and inspect the results.
- **Magnification At The Multiple Images:** The signed and unsigned magnification at every observed image position.
- **Chi Squared:** For point-source modeling, there are many different ways to define the likelihood function, broadly.
- **Solved Source Centre:** The `*Solved` fit variants solve the source-plane centre analytically instead of sampling it.
- **Fluxes:** Another measurable quantity of a point source is its flux—the total amount of light received from.
- **Flux Point Dataset:** The fluxes are not input a `PointDataset` object, alongside the image-plane coordinates of the.
- **Flux Fitting:** Above, we used a `FitPointDataset` to fit the positions of the point source in the image-plane.
- **Time Delays:** Another measurable quantity of a point source is its time delay—the time it takes for light to.
- **Time Delay Fitting:** We can also use the `FitPointDataset` to fit the time delays of the point source, which is done by.
- **New User Wrap Up:** The `point_source` package of the `autolens_workspace` contains numerous example scripts for.
- **Shape Solver:** All calculations above assumed the source was a point source with no size.

__JAX__

This guide walks through point-source fitting at a low level (no
non-linear search) — the `FitPointDataset` and `PointSolver` objects work
on either NumPy or JAX. For the standard search-driven modeling path
where `AnalysisPoint` auto-enables `use_jax=True` and the JIT happens
inside the search driver, see `start_here.py` / `modeling.py`. For the
explicit `@jax.jit + PointSolver(use_jax=True)` pattern (useful for fast
forward-solving in custom code), see `simulator.py`'s `__JAX Variant__`
section.

Note: `PointSolver(use_jax=True)` works inside `@jax.jit`. The triangle-
refinement loop operates on raw arrays and doesn't go through `.native`.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

from autolens import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path
import autofit as af
import autolens as al
import autolens.plot as aplt

__Lensed Point Source__

To begin, we create a strong lens image using an isothermal mass model and a source with a compact exponential light profile.

Although our goal is to demonstrate solving for the multiple image positions of a point source, simulating the data 
with a compact extended source makes the visualization of the point solver’s solutions clearer.

In [ ]:
grid = al.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.05,  # <- The pixel-scale describes the conversion from pixel units to arc-seconds.
)

isothermal_mass_profile = al.mp.Isothermal(
    centre=(0.0, 0.0),
    einstein_radius=1.6,
    ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
)

exponential_light_profile = al.lp.ExponentialCore(
    centre=(0.07, 0.07), intensity=0.1, effective_radius=0.1
)

lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=isothermal_mass_profile,
)

source_galaxy = al.Galaxy(
    redshift=1.0,
    light=exponential_light_profile,
)

tracer_extended = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

We plot the image of our strongly lensed source galaxy.

Clearly visible are four multiple images arranged in a cross configuration. The brightest pixels correspond to the 
four (y, x) multiple image positions that our point source solver aims to identify.

In [ ]:
aplt.plot_array(array=tracer_extended.image_2d_from(grid=grid), title="Image")

__Point Source__

The image above visually illustrates where the source’s light is traced on the image plane.

We now treat this source as a point source by defining a source galaxy using the `Point` class.

This point source shares the same center as the compact source above, ensuring that the multiple image positions 
coincide with those previously shown in the image plane.

In [ ]:
point_source = al.ps.Point(centre=(0.07, 0.07))

source_galaxy = al.Galaxy(redshift=1.0, point_0=point_source)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

__Point Solver__

For a point source, our goal is to find the (y, x) coordinates in the image plane that map directly to the center of 
the point source in the source plane—these are its "multiple images." This is achieved using a `PointSolver`, which 
determines the multiple images of the mass model for a point source located at a given (y, x) position in the 
source plane.

The solver works by ray tracing triangles from the image plane back to the source plane and checking whether the 
source-plane (y, x) center lies inside each triangle. It iteratively refines this process by ray tracing progressively 
smaller triangles, allowing the multiple image positions to be determined with sub-pixel precision.

The `PointSolver` requires an initial grid of (y, x) coordinates in the image plane, which defines the first set of 
triangles to ray trace. It also needs a `pixel_scale_precision` parameter, specifying the resolution at which the 
multiple images are computed. Smaller values increase precision but require longer computation times. The value 
of 0.001 used here balances efficiency and accuracy.

Strong lens mass models often predict a "central image," a multiple image that is usually heavily demagnified and thus 
not observed. Since the `PointSolver` finds all valid multiple images, it will locate this central image regardless of 
its visibility. To avoid including this unobservable image, we set a `magnification_threshold=0.1`, which discards any 
images with magnifications below this value.

If your dataset does include a detectable central image, you should lower this threshold accordingly to include it in 
your analysis.

We now compute the multiple image positions by creating a `PointSolver` object and passing it the tracer of our 
strong lens system.

In [ ]:
grid = al.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.2,  # <- The pixel-scale describes the conversion from pixel units to arc-seconds.
)

solver = al.PointSolver.for_grid(
    grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1
)

We now pass the tracer to the solver, to determine the image-plane multiple images for the source centre.

The solver will find the image-plane coordinates that map directly to the source-plane coordinate (0.07", 0.07"), 
which we plot below.

The plot shows the four solved multiple image positions (with the central image excluded) as a scatter plot. To make 
the positions clearer, we increase the marker size and use asterisks—PyAutoLens’s standard symbol for denoting 
multiple images of strong lenses.

In [ ]:
positions = solver.solve(tracer=tracer, source_plane_coordinate=(0.07, 0.07))

aplt.plot_grid(grid=positions, title="")

The plot above makes it difficult to directly compare the multiple image positions with the image of the strong lens itself.

To improve clarity, we overplot the multiple image positions on the strong lens image. This clearly shows that the 
multiple images coincide with the centers of the brightest pixels of the lensed source galaxy.

In [ ]:

aplt.plot_array(array=tracer_extended.image_2d_from(grid=grid), title="Image")

__Number of Solutions__

The number of solutions (e.g. the number of image-plane multiple images that map to the source centre) depends
on the mass model of the lens: 

 - For spherical mass profiles, there are three unique solutions, including a demagnified central image.

 - For elliptical mass profiles, there are five unique solutions, again including a demagnified central image.

 - For lenses with multiple mass profiles (e.g. two galaxies) and more exotic mass distributions, the number of 
   solutions can be even higher. 

__Solving the Lens Equation__

In the literature, the process of finding the multiple images of a source in the image-plane is often referred to as
'solving the lens equation'.

There lens equation is a fundamental equation in lensing, which describes how light rays are deflected from the
image-plane to the source-plane. It is given by:

$\beta = \theta - \hat{\alpha}(\theta)$

Where:

$\beta$ is the source-plane (y,x) coordinate.
$\theta$ is the image-plane (y,x) coordinate.
$\hat{\alpha}(\theta)$ is the deflection angle at image-plane (y,x) coordinate $\theta$.

The lens equation is non-linear, as the deflection angle $\hat{\alpha}$ depends on the mass model of the lens galaxy.

It is therefore called solving the lens equation because we are trying to find the image-plane (y,x) coordinates $\theta$
that satisfies the equation above for a given source-plane (y,x) coordinate $\beta$.

__Triangle Tracing__

Computing the multiple image positions of a point source is a non-linear problem. Given a source-plane (y,x) coordinate,
there are multiple image-plane (y,x) coordinates that trace to that source-plane coordinate, and there is no simple
analytic solution to determine these image-plane coordinates.

The solver therefore uses a triangulation approach to find the multiple image positions. It first overlays a grid of
triangles over the image-plane, and uses the mass model to trace these triangles to the source-plane. If a triangle
contains the source-plane (y,x) coordinate, it is retained and its image-plane coordinates are assigned as a multiple
image of the source.

We require the grid of triangles to be fine enough such that the source-plane (y,x) coordinate is contained within
one of the triangles to a sufficient precision for our science case. This is controlled by the `pixel_scale_precision`
input, which sets the target pixel scale of the grid. 

Triangles of iteratively finer resolution are created until this precision is met, therefore a lower value of
`pixel_scale_precision` will lead to a more precise estimate of the multiple image positions at the expense of
increased computational overhead.

Here is a visualization of the triangulation approach:

[CODE]

__Dataset__

We first create a `PointDataset` object, which is similar to an `Imaging` or `Interferometer` object but contains the
positions of the multiple images of the point source and their noise-map values.

The noise values are the centroid uncertainty with which each multiple image is localised in the data. This is *not*
the imaging pixel scale — that is the detector's sampling, not its centroiding precision. Bright point sources are
localised by fitting the instrumental PSF to the image, and the resulting centroid uncertainty is typically a small
fraction of a pixel: ~0.005" (5 mas) for HST or adaptive-optics imaging of lensed quasars in the strong-lensing
literature (CASTLES, TDCOSMO/H0LiCOW). See `scripts/point_source/simulator.py` for a full discussion.

We construct a `PointDataset` from the multiple-image positions we just solved for, with a small (5 mas) noise
value per position. In a real analysis the positions would come from your data reduction pipeline (e.g. a
centroid fit to each multiple image in the calibrated image), but for this demo we reuse the solver's output
so the script adapts to whatever number of multiple images the mass model produces.

The demagnified central image is excluded by the solver's `magnification_threshold=0.1` setting, so it does not
appear in the dataset either — standard practice in point-source modeling.

The dataset's name `point_0` is an important label, as explained in more detail below.

In [ ]:
positions_data = al.Grid2DIrregular(positions)

positions_noise_map = al.ArrayIrregular([0.005] * len(positions))

dataset = al.PointDataset(
    name="point_0", positions=positions_data, positions_noise_map=positions_noise_map
)

We can print this dictionary to see the dataset's `name`, `positions` and `fluxes` and noise-map values.

In [ ]:
print("Point Dataset Info:")
print(dataset.info)

The positions can be plotted over the observed image, to make sure they overlap with the multiple images we expect.

In [ ]:

aplt.plot_array(array=tracer_extended.image_2d_from(grid=grid), title="Image")

__Name Pairing__

The names of the point-source datasets have an even more important role, the names are used to pair each dataset to the
point sources in the lens model used to fit it.

For example, when creating the tracer at the beginning of this script, we named the point source `point_0`:

point_source = al.ps.Point(centre=(0.07, 0.07))
source_galaxy = al.Galaxy(redshift=1.0, point_0=point_source)
tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

When we fit the point source dataset using this tracer, the name is again used in order to pair the dataset to the
this point source. This means that point source with a centre of (0.07", 0.07") is used to fit the dataset with the
name `point_0`.

If there is no point-source in the model that has the same name as a `PointDataset`, that data is not used in
the model-fit. If a point-source is included in the model whose name has no corresponding entry in 
the `PointDataset` an error will be raised.

In this example, where there is just one source, name pairing is redundant. However, point-source datasets may
have many source galaxies in them, and name pairing allows us to extend the point-source modeling to systems with
many point sources.

__Fitting__

Just like we used a `Tracer` to fit imaging and interferometer data, we can use it to fit point-source data via the
`FitPoint` object.

The name pairing described above is used internally into the `FitPointDataset` object to ensure that the correct point
source is fitted to each dataset. 

The fit is returned as a dictionary which mirrors the `PointDataset`, where its keys are again the names of the datasets.

In [ ]:
fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsImagePairRepeat,  # This input is describe below
)

print(fit.positions.residual_map)
print(fit.positions.normalized_residual_map)
print(fit.positions.chi_squared_map)
print(fit.positions.log_likelihood)

__Magnification At The Multiple Images__

The fit also gives the magnification of the tracer at every observed image position, computed via the Hessian as
`mu = 1 / det(A)`, where `A` is the lensing Jacobian.

`signed_magnifications_at_positions` keeps the sign, which is the parity of each image: positive for images at
minima (or maxima) of the arrival-time surface and negative for saddle-point images. `magnifications_at_positions`
is its absolute value, which is the magnification used when fitting the fluxes of the multiple images (see
`__Flux Fitting__` below).

For a multi-plane tracer, these are computed up to the plane of the point source paired with the dataset, so in a
cluster each source gets the magnification at its own redshift. A finished model-fit gives the same quantities for
the maximum likelihood model via `result.signed_magnifications_at_positions` and
`result.magnifications_at_positions` (see `point_source/modeling.py`).

To compute the magnification at any other (y,x) coordinates, use
`al.LensCalc.from_tracer(tracer=tracer).magnification_2d_via_hessian_from(grid=al.Grid2DIrregular(...))`, which is
described in `autolens_workspace/*/guides/lens_calc.py`.

In [ ]:
print(fit.positions.signed_magnifications_at_positions)
print(fit.positions.magnifications_at_positions)

__Chi Squared__

For point-source modeling, there are many different ways to define the likelihood function, broadly referred to a
an `image-plane chi-squared` or `source-plane chi-squared`. This determines whether the multiple images of the point
source are used to compute the likelihood in the source-plane or image-plane.

The default settings used above use the image-plane chi-squared, which uses the `PointSolver` to determine the 
multiple images of the point source in the image-plane for the given mass model and compares the positions of these 
model images to the observed images to compute the chi-squared and likelihood.

There are still many different ways the image-plane chi-squared can be computed, differing in how model images
are paired to observed positions: do we allow repeat image-pairs (the same model image matched to multiple
observed positions)? Do we pair all possible combinations of model images to observed positions? The default is
the all-to-all pairing combined with the analytically-solved source centre (`FitPositionsImagePairAllSolved`,
illustrated in the `__Solved Source Centre__` section below); the examples here walk through the free-centre
pairings one by one, passing each class explicitly.

For example, we can repeat the fit above whilst not allowing for repeat image pairs as follows:

In [ ]:
fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsImagePair,  # Different input to the one used above
)

print(
    "Minimum Distance Between Observed Multiple Images and Model Multiple Images Without Repeats:"
)
print(fit.positions.residual_map)

print("Log Likelihood Without Repeats:")
print(fit.positions.log_likelihood)

We can allow for repeat image pairs by using the `FitPositionsImagePairRepeat` class.

In [ ]:
fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsImagePairRepeat,  # Different input to the one used above
)

print(
    "Minimum Distance Between Observed Multiple Images and Model Multiple Images With Repeats:"
)
print(fit.positions.residual_map)

print("Log Likelihood With Repeats:")
print(fit.positions.log_likelihood)

The `FitPositionsImagePairAll` class instead marginalizes each observed position over *every* model
image via a smooth probabilistic mixture — the pairing scheme of the default fit (in its solved-centre
form). Its key property is robustness: if a true multiple image is missing from the dataset (e.g.
hidden under the lens galaxy's light), the model image that would have paired to it is left as the
nearest neighbour of no observed position, so repeat pairing's `unmatched_model_policy` charges it
as a bright extra image and mis-ranks the true model catastrophically — while the all-to-all
mixture absorbs the missing image gracefully as a mild Occam factor. On clean data the two give statistically equivalent
results, which is why robustness decides the default.

In [ ]:
fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsImagePairAll,  # Different input to the one used above
)

print("Log Likelihood With All-To-All Pairing:")
print(fit.positions.log_likelihood)

For a "source-plane chi-squared", the likelihood is computed in the source-plane. The analysis is simpler, it ray-traces
the multiple images back to the source-plane and measures the distance of each traced image from the point source 
centre. 

By default (`weighting = "jacobian"`) each distance is measured in the image-plane noise ellipse: the lensing Jacobian 
at each image maps the source-plane residual back to the image plane, where the position noise is defined. This is 
the correct linearised likelihood, it stays robust near critical curves and it costs no more than weighting by the 
magnification. The older scalar weighting, `weighting = "magnification"`, is an opt-in that is biased near critical 
curves. Both are explained in `guides/point_solver/pairing.py`.

The source-plane chi-squared is significantly faster to compute than the image-plane chi-squared, however it cannot 
tell when a model predicts extra images and is only a first-order approximation to the image-plane chi-squared, so 
it can still bias lens model results. 

Here is an example of how to use the source-plane chi-squared:

In [ ]:
fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsSource,  # Different input to the one used above
)

print(
    "Minimum Distance Between Source Plane Centre and Model Source Plane Images After Ray Tracing:"
)
print(fit.positions.residual_map)

print("Log Likelihood in the Source Plane:")
print(fit.positions.log_likelihood)

A note on defaults: `FitPointDataset` and `AnalysisPoint` — the object used in the model-fitting
examples — both default to `FitPositionsImagePairAllSolved` (all-to-all pairing with the analytically
solved source centre, demonstrated below). The examples above pass `fit_positions_cls` explicitly so
there is no ambiguity about which chi-squared is being used — note the free-centre classes used here
require a centre-bearing profile (`al.ps.Point`), while the solved default requires the parameter-free
`al.ps.PointSolved`.

__Solved Source Centre__

Every fit above reads the source-plane centre from the model's `Point` profile, whose `centre` is a
free parameter during model-fitting. Each fit class also has a `*Solved` variant which instead solves
the centre analytically from the observed positions and the mass model, removing those 2 free
parameters per point source from the non-linear search: `FitPositionsSourceSolved` (a
precision-weighted source-plane centre, following Lombardi 2024, arXiv:2406.15280) and
`FitPositionsImagePairRepeatSolved` / `FitPositionsImagePairAllSolved` (the same solved centre
driving the image-plane forward solve).

The solved variants pair with the parameter-free `al.ps.PointSolved` model component instead
of `Point` — mixing the two conventions (a `Solved` fit with a `Point`, or vice versa) raises
a `PointProfileMismatchException`, so a model cannot silently sample centre parameters the fit
ignores.

In [ ]:
point_source_solved = al.ps.PointSolved()

source_galaxy_solved = al.Galaxy(redshift=1.0, point_0=point_source_solved)

tracer_solved = al.Tracer(galaxies=[lens_galaxy, source_galaxy_solved])

fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer_solved,
    solver=solver,
    fit_positions_cls=al.FitPositionsSourceSolved,  # Solved-centre source-plane chi-squared
)

print("Analytically Solved Source-Plane Centre:")
print(fit.positions.source_plane_coordinate)

print("Log Likelihood with Solved Centre:")
print(fit.positions.log_likelihood)

The full matrix of pairing schemes × centre treatments — including the solved flux and time-delay
variants used below, the `"jacobian"` and `"magnification"` source-plane weightings, and when each option is
preferable — is documented in `guides/point_solver/pairing.py`.

__Fluxes__

Another measurable quantity of a point source is its flux—the total amount of light received from each multiple image 
of the point source (e.g., the quasar images).

In practice, fluxes are often measured but not used directly when analyzing lensed point sources such as quasars or 
supernovae. This is because fluxes can be significantly affected by microlensing, which many lens models do not 
accurately capture. However, in this simulation, microlensing is not included, so the fluxes can be simulated and 
fitted reliably.

We now simulate the fluxes of the multiple images of this point source.

Given a mass model and the (y, x) image-plane coordinates of each image, the magnification at each point can be 
calculated.

Below, we compute the magnification for every multiple image coordinate, which will then be used to simulate their 
fluxes.

In [ ]:
magnifications = al.LensCalc.from_tracer(
    tracer=tracer
).magnification_2d_via_hessian_from(grid=positions)

To simulate the fluxes, we assume the source galaxy point-source has a total flux of 1.0.

Each observed image has a flux that is the source's flux multiplied by the magnification at that image-plane coordinate.

In [ ]:
flux = 1.0
fluxes = [flux * np.abs(magnification) for magnification in magnifications]
fluxes = al.ArrayIrregular(values=fluxes)

We set the flux noise to 5% of each measured flux. For lensed quasars and supernovae, photometric flux
uncertainties are dominated by microlensing systematics rather than photon noise, so models that exclude
microlensing typically assume a few-percent flux uncertainty per image. See `scripts/point_source/simulator.py`
for a fuller discussion.

In [ ]:
flux_rel_noise = 0.05

fluxes_noise_map = al.ArrayIrregular(values=flux_rel_noise * np.asarray(fluxes))

__Flux Point Dataset__

The fluxes are not input a `PointDataset` object, alongside the image-plane coordinates of the multiple images
and their associated noise-map values. 

We again give the dataset the name `point_0`, which is a label given to the dataset to indicate that it is a dataset 
of a single point-source.

In [ ]:
dataset = al.PointDataset(
    name="point_0",
    positions=positions_data,
    positions_noise_map=positions_noise_map,
    fluxes=fluxes,
    fluxes_noise_map=fluxes_noise_map,
)

__Flux Fitting__

Above, we used a `FitPointDataset` to fit the positions of the point source in the image-plane.

We can also use it to fit the fluxes of the point source, which is done by passing the new dataset also containing
the `fluxes` and `fluxes_noise_map` to the fit.

To fit fluxes, our model point source also needs a flux parameter, which is done by using the `PointFlux`
component instead of the `Point` component. 

In [ ]:
point = al.ps.PointFlux(centre=(0.07, 0.07), flux=1.0)

source_galaxy = al.Galaxy(redshift=1.0, point_0=point)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsImagePairRepeat,  # This input is describe below
)

The fit now contains both a `positions` and `fluxes` attribute, which contain the fit of the positions and fluxes
of the point source.

In [ ]:
print(fit.positions.residual_map)
print(fit.positions.normalized_residual_map)
print(fit.positions.chi_squared_map)
print(fit.positions.log_likelihood)

print(fit.flux.residual_map)
print(fit.flux.normalized_residual_map)
print(fit.flux.chi_squared_map)
print(fit.flux.log_likelihood)

An analytic-flux variant also exists: `fit_flux_cls=al.FitFluxesSolved` (paired with
`al.ps.PointSolved`) solves the source flux from the data and magnifications instead of sampling
a `flux` parameter — see `point_source/features/fluxes.py` and `guides/point_solver/pairing.py`.

__Time Delays__

Another measurable quantity of a point source is its time delay—the time it takes for light to travel from the
source to the observer for each multiple image of the point source (e.g., the quasar images). This is often expressed
as the relative time delay between each image and the image with the shortest time delay, which is often referred to as
the "reference image."

Time delays are commonly used in strong lensing analyses, for example to measure the Hubble constant, since
they are less affected by microlensing and can provide robust cosmological constraints.

We now simulate the same point source dataset, but this time including the time delays of the multiple images.

Given a mass model and (y, x) image-plane coordinates, the time delay at each image-plane position can be
calculated from the mass model. It includes the contribution of both the geometric time delay (the time it takes
different light rays to travel from the source to the observer) and the Shapiro time delay (the time it takes
light to travel through the gravitational potential of the lens galaxy).

In [ ]:
time_delays = tracer.time_delays_from(grid=positions)

In real observations, time delays are measured by photometrically monitoring the multiple images over months
to years and cross-correlating their light curves. State-of-the-art monitoring campaigns (COSMOGRAIL, TDCOSMO)
achieve ~1–3% relative precision on the longest delays in well-sampled quad systems. For simplicity we adopt
a 5% relative uncertainty here. Real-world uncertainties are not strictly proportional to the delay magnitude
(they depend on monitoring cadence, length, and microlensing variability), but a constant fractional error is
a reasonable simulator default. See `scripts/point_source/simulator.py` for a fuller discussion.

In [ ]:
time_delay_rel_noise = 0.05

time_delays_noise_map = al.ArrayIrregular(
    values=np.abs(time_delays) * time_delay_rel_noise
)

The time delays are input into a `PointDataset` object, alongside the image-plane coordinates of the multiple images
and their associated noise-map values. 

We again give the dataset the name `point_0`, which is a label given to the dataset to indicate that it is a dataset 
of a single point-source.

In [ ]:
dataset = al.PointDataset(
    name="point_0",
    positions=positions_data,
    positions_noise_map=positions_noise_map,
    time_delays=time_delays,
    time_delays_noise_map=time_delays_noise_map,
)

__Time Delay Fitting__

We can also use the `FitPointDataset` to fit the time delays of the point source, which is done by passing the new
dataset also containing the `time_delays` and `time_delays_noise_map` to the fit.

To fit time delays, the model point source does not need any special parameters (like it did for flux fitting),
so we can revert back to the normal `Point` component.

In [ ]:
point = al.ps.Point(centre=(0.07, 0.07))

source_galaxy = al.Galaxy(redshift=1.0, point_0=point)

tracer = al.Tracer(galaxies=[lens_galaxy, source_galaxy])

fit = al.FitPointDataset(
    dataset=dataset,
    tracer=tracer,
    solver=solver,
    fit_positions_cls=al.FitPositionsImagePairRepeat,  # This input is describe below
)

The fit now contains both a `positions` and `time_delays` attribute, which contain the fit of the positions and fluxes
of the point source.

In [ ]:
print(fit.positions.residual_map)
print(fit.positions.normalized_residual_map)
print(fit.positions.chi_squared_map)
print(fit.positions.log_likelihood)

print(fit.time_delays.residual_map)
print(fit.time_delays.normalized_residual_map)
print(fit.time_delays.chi_squared_map)
print(fit.time_delays.log_likelihood)

As with fluxes, an analytic variant exists: `fit_time_delays_cls=al.FitTimeDelaysSolved` replaces
the reference-image subtraction with a precision-weighted analytic reference time — see
`point_source/features/time_delays.py` and `guides/point_solver/pairing.py`.

__New User Wrap Up__

The `point_source` package of the `autolens_workspace` contains numerous example scripts for performing point source
modeling. These focus on "galaxy scale" lenses, which are lenses that have a single lens galaxy, as opposed to
"group scale" or "cluster scale" lenses which have multiple lens galaxies.

Point source modeling is at the heart of group and cluster scale lens modeling, and is the topic of the
next overview script.

__Shape Solver__

All calculations above assumed the source was a point source with no size. 

This was built into the point-solver, for example when we solved for the multiple images of the point source in the 
image-plane, we ray-traced triangles to the source-plane and asked whether the source-plane (y,x) centre was within 
the triangle.

There is functionality to include the size and shape of the source in the calculation, which uses the `ShapeSolver`
class. This still traces triangles, but each iteration of the solver now computes the area of each image-plane triangle 
that is within the source-plane shape. This means we can determine the area in the image-plane that maps within an 
extended region of the source-plane shape.

For example, by inputting the shape `Circle` with a radius of 0.001", the shape solver will determine the area of the 
multiple images pixel which fall within this circle, which is different information to the point solver which told
us the exact (y,x) coordinates of the multiple images.

The ratio of the total image pixel area to the area within the source-plane 
circle is the magnification factor of the source. This magnification factor then changes the observed flux of each 
multiple image.

Observations we might think are fully in the point source regime therefore may have an observable signature of the size
of the source in the flux ratios and magnifications of the multiple images. Therefore, sometimes the source size 
is large enough that it is important we account for it.

In [ ]:
solver = al.ShapeSolver.for_grid(
    grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1
)

solver.find_magnification(tracer=tracer, shape=al.Circle(x=0.0, y=0.0, radius=0.001))
